# Merge the predicted results back to the original CRS 

In [1]:
import pandas as pd

# Load the CRS data from parquet file
crs = pd.read_parquet("../../data/raw/CRS.parquet")

In [ ]:
# Read minig_sets from feather file
stat_mining_set = pd.read_feather("../../data/processed/prediction_sets/stat_to_mine.feather")
gen_mining_set = pd.read_feather("../../data/processed/prediction_sets/gen_to_mine.feather")

# Load the unlabled_predicted data from feather file
unlabeled_predicted_stat = pd.read_feather("../../data/processed/predicted/unlabeled_predicted_stat.feather")
unlabeled_predicted_gen = pd.read_feather("../../data/processed/predicted/unlabeled_predicted_gen.feather")

# Load conflicting_descr_stat_predicted from feather file
conflicting_stat_predicted = pd.read_feather("../../data/processed/predicted/conflicting_stat_predicted.feather")
conflicting_gen_predicted = pd.read_feather("../../data/processed/predicted/conflicting_gen_predicted.feather")

Add keywords back to the raw CRS data

In [5]:
# Load the matched CRS data from feather file
crs_title_matched = pd.read_feather("../../data/processed/title_matched/crs_titles_matched_wo_stopwords.feather")

# Keep only the columns project_title, stat_keywords, stat_blacklist, gen_keywords, stat_acronyms, gen_acronyms
crs_title_matched = crs_title_matched[['project_title', 'language', 'stat_keywords', 'stat_blacklist', 'gen_keywords', 'stat_acronyms', 'gen_acronyms']]

# Rename language to title_language
crs_title_matched.rename(columns={'language': 'title_language'}, inplace=True)

# Left join matched CRS data to the the raw CRS data based on the project_title column
crs = pd.merge(crs, crs_title_matched, on='project_title', how='left', validate='m:1')

del crs_title_matched

In [6]:
# Add predicted probabilities of unlabeled predicted set to mining sets
stat_mining_predicted = pd.merge(
    stat_mining_set,
    unlabeled_predicted_stat[['text_mining_description', 'probability_is_statistics']],
    on='text_mining_description',
    how='left'
)

gen_mining_predicted = pd.merge(
    gen_mining_set,
    unlabeled_predicted_gen[['text_mining_description', 'probability_is_gender']],
    on='text_mining_description',
    how='left',
    validate='m:1'
)

del stat_mining_set, unlabeled_predicted_stat, gen_mining_set, unlabeled_predicted_gen

stat_mining_predicted.loc[stat_mining_predicted['is_mining'] == True, 'probability_is_statistics'] = 0

In [7]:
# Only keep text_mining_description and probability columns
stat_mining_predicted = stat_mining_predicted[['text_mining_description', 'probability_is_statistics']]
gen_mining_predicted = gen_mining_predicted[['text_mining_description', 'probability_is_gender']]

# Concanate rows of conflicting predicted to stat/gen_mining_predicted
stat_mining_predicted = pd.concat([stat_mining_predicted, conflicting_stat_predicted], ignore_index=True)
gen_mining_predicted = pd.concat([gen_mining_predicted, conflicting_gen_predicted], ignore_index=True)

In [10]:
# Replace missing values or "" of long_descriptions with "N/A", make temporary column for preserving long_description
crs['long_description_for_mining'] = crs['long_description'].replace("", "N/A")
crs['long_description_for_mining'] = crs['long_description_for_mining'].fillna("N/A")

# Create text mining descripton: join short_description, ": ", long_description 
crs['text_mining_description'] = crs['short_description'].astype(str) + ": " + crs['long_description_for_mining'].astype(str)

# Drop long_description_for_mining column
crs.drop(columns=['long_description_for_mining'], inplace=True)

In [11]:
# Merge the stat_mining_predicted with the CRS data on 'text_mining_description'
crs = pd.merge(
    crs,
    stat_mining_predicted[['text_mining_description', 'probability_is_statistics']],
    on='text_mining_description',
    how='left',
    validate='m:1'
)

crs = pd.merge(
    crs,
    gen_mining_predicted[['text_mining_description', 'probability_is_gender']],
    on='text_mining_description',
    how='left',
    validate='m:1'
)

del stat_mining_predicted, gen_mining_predicted

#### Set probabilities according to keywords and purpose codes

In [13]:
import numpy as np
# Overwrite the probability_is_statistics from original title matches for projects in conflicting_descr_stat that had a None title and statistics title with the same text_minig_description 
# IF purpose_code == 16062 | stat_keywords is not null | stat_acronyms is not null & stat_blacklist is null & pupose_code != 15250 & pupose_code != 93010, set probability_is_statistics to 1
crs.loc[
    (
        (crs['purpose_code'] == 16062) |    # purpose_code is 16062
        (crs['stat_keywords'].notna()) |    # stat_keywords is not null
        (crs['stat_acronyms'].notna())      # stat_acronyms is not null
    ) &
    (crs['stat_blacklist'].isna()) &        # stat_blacklist must be null
    (crs['purpose_code'] != 15250) &        # purpose_code must not be 15250 
    (crs['purpose_code'] != 93010),         # purpose_code must not be 93010
    'probability_is_statistics'
] = 1

# Set probability to 0 for mining projects
crs.loc[
    (crs['purpose_code'] == 15250) |        # purpose_code is 15250
    (crs['purpose_code'] == 93010) |        # purpose_code is 93010
    (crs['stat_blacklist'].notna()),        # keyword in blacklist
    'probability_is_statistics'
] = 0

# Manual corrections after inspection of results of title matching:
#       - Set is_statistics to 0 for title_language == it and stat_acronyms == ' ai ' 
#       - Set is_statistics to 0 for title_language == it and stat_acronyms == ' ia ' prior to 2017
crs.loc[
    (crs['title_language'] == 'it') & 
    (crs['stat_acronyms'].apply(lambda x: isinstance(x, (list, np.ndarray)) and ' ai ' in x)), 
    'probability_is_statistics'
] = 0

# Set is_statistics to 0 for year <= 2017 and stat_acronyms contains ' ai ' or ' ia ', as all AI was exclusively used for other purposes
crs.loc[
    (crs['year'] <= 2017) & 
    ((crs['stat_acronyms'].apply(lambda x: isinstance(x, (list, np.ndarray)) and ' ai ' in x)) |
    (crs['stat_acronyms'].apply(lambda x: isinstance(x, (list, np.ndarray)) and ' ia ' in x))),
    'probability_is_statistics'
] = 0

In [14]:
# Make gen_donor as (donor_code == 1 & agency_code == 16) | (donor_code == 5 & agency_code == 65)
crs['gen_donor'] = (
    ((crs['donor_code'] == 1) & (crs['agency_code'] == 16)) |   # donor_code is 1 and agency_code is 16 -> Austrian Federal Ministry of Education and Women's Affairs
    ((crs['donor_code'] == 5) & (crs['agency_code'] == 65))     # donor_code is 5 and agency_code is 65 -> German Federal Ministry for Family Affairs, Senior Citizens, Women and Youth
)

# Set gen_donor to False in case gen_donor is null
crs['gen_donor'] = crs['gen_donor'].fillna(False)

# Gender filter 
crs.loc[
    (crs['purpose_code'].between(15170, 15180)) |                       # purpose_code is in 15170:15180
    (crs['gender'] == 2) |                                              # gender is 2
    (crs['gen_keywords'].notna()) |                                     # gen_keywords is not null
    (crs['gen_acronyms'].notna()) |                                     # gen_acronyms is not null
    (crs['rmnch'] == 2) |                                               # rmnch is 2
    (crs['channel_code'].isin([41146, 21053, 21040, 21037, 21010])) |   # channel_code 
    (crs['gen_donor']),                                                 # gen_donor is True
    'probability_is_gender'
] = 1

In [ ]:
# How many rows have NaN in probability_is_statistics
crs['probability_is_statistics'].isna().sum()

In [15]:
# Check if there are any projects without a probability estimate
no_prob_stat = crs[crs['probability_is_statistics'].isna()]
no_prob_gen = crs[crs['probability_is_gender'].isna()]

In [ ]:
# WARNING: uncomment only if projects with no probability inspected and made sure that they can be set to 0
# Set rest of the probability_is_statistics to 0
# crs['probability_is_statistics'] = crs['probability_is_statistics'].fillna(0)

In [ ]:
# Save as crs_predicted.feather
crs.to_feather("../../data/output/crs_predicted.feather")